# Anatomy of Horror  
## Horror Films as a Mirror of Collective Fear

---

## 1. Introduction
This project explores horror films as a cultural barometer by analyzing whether relevant patterns can be identified in their stories and visual languages.
<br>
The horror genre is a particularly useful prism for analyzing collective anxieties and fears. Beneath the variety of monsters and myths, horror films keep returning to the same core primal fears, recurring across time and context: anxieties about who we are, what we control, and how easily both can be taken away.

We analyze the horror genre as a dynamic system of motifs, tropes, and aesthetic patterns by combining structured metadata, language models, and unsupervised image analysis.

To pursue this goal, we take two parallel yet interconnected approaches.
1. Narrative analysis, which focuses on plot summaries and metadata
2. Visual analysis, which focuses on horror film posters and aesthetic clustering.

This process was conducted across **three notebooks**:
- **horror_project.ipynb**
- **posters_analysis1.ipynb**
- **posters_analysis2.ipynb**

### Expectations & Results
Going into the project, I did not expect the poster-clustering pipeline to perform as effectively as it did. I initially assumed that CLIP embeddings would primarily group posters based on features like color palettes, typography, and overall design conventions, rather than capturing deeper thematic structure. Instead, the resulting clusters consistently reflected not only shared aesthetic patterns but also recurring iconography (e.g., creatures, religious imagery, specific isolation or survival settings), as well as decade-specific visual styles. 

I also expected TF–IDF to surface clearer narrative signals for a larger proportion of clusters. While it proved effective as a validation tool in several cases, the short and formulaic nature of many plot overviews (and the fact that a large number of clusters inevitably includes noisier, more heterogeneous groups) likely limited the consistency of the extracted keywords. Even so, TF–IDF often helped corroborate the narrative content suggested by the posters, and in some cases also helped identify the core common denominator in clusters that were visually apparent but difficult to define precisely. 

Finally, the LLM-based fear classifier performed better than I expected: after several iterations in the prompt, it produced labels that were mostly accurate and coherent to allow for historical scaling. To fully leverage this methodology, a natural next step would be to conduct a region- or country-specific analysis to examine whether specific cultural or historical shocks correlate with changes in the fears depicted on screen.

---

## Pipeline Overview

### Process
The pipeline runs along two parallel tracks that then converge for cross-interpretation. First, we collect structured metadata from TMDb and IMDb (titles, plot overviews, keywords, ratings, countries, etc.) and use a local LLM (LLaMA 3 via Ollama) to assign each film to one of 11 fear categories, manually defined. This produces a scalable and internally consistent labeling system. In parallel, we download poster images and embed them with CLIP to represent each poster as a 512-dimensional vector. We then reduce dimensionality with PCA (280 components) and cluster posters with KMeans (k=50) to identify recurring visual aesthetic patterns. Finally, we merge these outputs to examine how visual motifs align (or don't) with fear categories and decades, and we use TF-IDF to analyze patterns in the overview texts within clusters as an additional interpretive check.


```text
IMDb / TMDb metadata (through API)
|
|-> movie names, plot overviews, keywords -> LLM Fear Classification
|
|-> poster images -> CLIP embeddings -> PCA (280) -> KMeans (50)
                                                        |
                                                        v
                                                poster clusters
                                                        |
                        -------------------------------------------------
                        |                           |                   |
                cluster × fear categories / cluster × decades / cluster × overview text
                                                                        |
                                                                clusters re-validation


---
## Phase 1 — Situating Horror Within the Genre Landscape (IMDb, 2020–2024)

### Goal
To contextualize horror in relation to other genres in terms of production volume, audience engagement, and critical acclaim.

### Process
- Source: IMDb public TSV datasets (`title.basics`, `title.ratings`)
- Filtered to:
  - films
  - released between 2020 and 2024
- Selected the top 3,000 films by vote count
- Mapped IMDb IDs to TMDb IDs to enrich genre metadata

### Technical Decisions
- **Temporal restriction (2020–2024)** avoids bias towards older movies in vote accumulation
- **Vote count** used as an imperfect proxy for cultural visibility
- **Multi-genre films** counted fully for each genre (e.g., if a movie is labeled as horror, comedy, and drama, it will count fully for all three genres)
Note: Because films can belong to multiple genres, genre totals are not mutually exclusive, so this analysis is descriptive rather than an accurate market-share estimate.

### Key Outcomes
| Genre | Avg Rating | Total Popularity Votes | Number of Titles |
|-------|------------|------------------------|------------------|
| Documentary | 7.27 | 1,697,289 | 143 |
| Music | 6.93 | 2,374,578 | 95 |
| History | 6.84 | 8,470,970 | 190 |
| Animation | 6.77 | 7,006,518 | 158 |
| War | 6.69 | 2,804,552 | 67 |
| Drama | 6.49 | 41,677,072 | 1,426 |
| TV Movie | 6.29 | 195,139 | 16 |
| Fantasy | 6.29 | 13,759,170 | 254 |
| Family | 6.29 | 7,728,394 | 200 |
| Crime | 6.22 | 16,970,079 | 432 |
| Adventure | 6.19 | 27,991,340 | 321 |
| Romance | 6.19 | 10,930,888 | 417 |
| Comedy | 6.19 | 31,576,520 | 886 |
| Western | 6.04 | 671,605 | 24 |
| Mystery | 5.98 | 10,836,194 | 293 |
| Action | 5.97 | 39,446,675 | 702 |
| Thriller | 5.91 | 33,015,636 | 957 |
| Science Fiction | 5.87 | 22,184,824 | 287 |
| Horror | 5.54 | 17,242,063 | 509 |


Horror sits at the edge of mainstream, between mass appeal and critical rejection. Despite its significant popularity, it has the lowest average rating of all genres, positioning itself as a genre with high consumption and low prestige. This makes horror particularly interesting as a "cultural mirror," functioning as a kind of popular unconscious rather than an arthouse, author-driven format.

---
## Phase 2 — Constructing a Long-Run Horror Corpus (TMDb, 1950–2024)

### Goal
To build a large, temporally partially balanced dataset of horror films suitable for conducting historical analysis.

### Process
- Used TMDb’s API to retrieve all films tagged as horror
- Assigned each film to a decade (1950s–2020s)

### Technical Decisions
- **Decade-based balancing** to prevent modern decades from overwhelming older ones, given that overall film production has grown over time
<br>
Hybrid sampling criteria: Each decade is assigned a base quota of 280 films, with additional slots from the 5,000 total distributed proportionally to each decade's total production volume. Within each decade, films are ranked by TMDb popularity and the top N films are selected. Because selection within each decade is popularity-weighted, the corpus reflects culturally visible films rather than a representative sample of all horror releases.
- Final dataset size: 5,000 films


---

## Phase 3 — LLM-Assisted Fear Classification
### Goal
To classify horror films by their **dominant fear**.

### Process
- Input per film:
  - title
  - plot overview
  - keyword metadata
- Model:
  - LLaMA 3 (8B) running locally via Ollama
- Prompt required:
  - selection of one fear category from the set of 11 existing, manually defined ones
  - justification for the decision

Note on classification validity. To evaluate whether the fear taxonomy could be applied reliably at scale, I conducted multiple test runs on a 30-film validation set covering all 11 fear categories, refining the prompt until the model consistently classified all cases correctly. The finalized prompt was then applied to the full corpus. While LLM-based classification is not perfect (nor would human annotation be, given that fear is inherently subjective and often overlapping) the LLM allows for a scalable and internally consistent labeling framework.


### Key Outcomes

| Fear Category | Count | Percentage |
|-----------|-------|------------|
| Invasion & Paranoia | 1377 | 27.95% |
| Captivity & Voyeuristic Sadism | 863 | 17.52% |
| Possession & Loss of Agency | 719 | 14.59% |
| Grief & Familial Trauma | 437 | 8.87% |
| Body Horror | 398 | 8.08% |
| Institutional & Structural Control | 336 | 6.82% |
| Persecution & Social Breakdown | 278 | 5.64% |
| Ecological / Natural Menace | 184 | 3.73% |
| Isolation & Psychological Unraveling | 155 | 3.15% |
| Contagion & Mutation | 117 | 2.37% |
| Transgression & Moral Punishment | 63 | 1.28% |

![graph1](assets/graph1.png)
![graph2](assets/graph2.png)

Invasion & Paranoia (28%) is the dominant fear, followed by Captivity & Voyeuristic Sadism (17.5%) and Possession & Loss of Agency (14.6%).

Among the key temporal trends, Invasion & Paranoia appears highest in the 1950s, declined mid-century, then rises sharply again in the 2000s-2010s. Captivity rose sharply in the 1970s and maintained somewhat consistent levels in the decades that followed. Possession peaked in the 1990s, dropped in the 2000s, then rebounds in the 2010s-2020s. Though still a small subgroup, Isolation grows steadily post-2000s.


---

## Phase 4 — Poster Image Collection and Preprocessing

### Goal
To assemble a large, standardized visual dataset of horror movie posters.

### Process
- Downloaded poster images using TMDb `poster_path`
- Save images locally

### Technical Decisions
- Images resized at load time
- Checkpointing used to ensure safe re-runs

### Key Outcomes
A total of 4,928 posters downloaded successfully

---

## Phase 5 — Visual Embeddings with CLIP

### Goal
To represent poster images numerically to capture relevant visual information that will be used as input for clustering.

### Process
- Model:
  - **CLIP ViT-B/32**
- Each poster embedded into a **512-dimensional vector**
- Stored embeddings and index mappings locally

### Technical Decisions
- CLIP chosen for its ability to encode high visual complexity

### Key Outcomes
Embeddings generated for the 4,928 posters, saved in a matrix of shape (4928, 512)


---
## Phase 6 — Dimensionality Reduction and Clustering

### Goal
To identify recurring visual clusters in horror posters.

### Process
- PCA applied to CLIP embeddings (experimented with multiple dimension values)
- KMeans clustering (experimented with multiple k values)
- Selected experiment:
  - PCA = 280 components
  - KMeans = 50 clusters

### Technical Decisions
- PCA reduces dimensionality and often can improve clusters by concentrating variance in fewer components
- KMeans chosen for the possibility to experiment with explicit cluster numbers
- Random seeds fixed for PCA/KMeans to improve reproducibility

### Key Outcomes
![Cluster 1](assets/cluster_1.png)
![Cluster 3](assets/cluster_3.png)
![Cluster 8](assets/cluster_8.png)
![Cluster 9](assets/cluster_9.png)
![Cluster 14](assets/cluster_14.png)
![Cluster 16](assets/cluster_16.png)
![Cluster 22](assets/cluster_22.png)
![Cluster 25](assets/cluster_25.png)
![Cluster 28](assets/cluster_28.png)
![Cluster 30](assets/cluster_30.png)
![Cluster 31](assets/cluster_31.png)
![Cluster 39](assets/cluster_39.png)
![Cluster 41](assets/cluster_41.png)
![Cluster 42](assets/cluster_42.png)
![Cluster 43](assets/cluster_43.png)
![Cluster 46](assets/cluster_46.png)
![Cluster 47](assets/cluster_47.png)
![Cluster 48](assets/cluster_48.png)

CLIP embeddings combined with PCA and KMeans produced a meaningful set of poster patterns, demonstrating that this approach is a valuable tool for this type of exploration.

---
## Phase 7 — Cross-Validation of Visual Clusters

### Goal
To test whether image-based poster clusters align with narrative fear categories and temporal patterns.

### Process
**Step 1: Clusters × Fear Categories**
- Merged cluster assignments with fear classification labels
- Restricted analysis to clusters that were visually coherent enough to label during manual review
- Calculated top 3 fear categories per cluster

**Step 2: Clusters × Decades**
- Analyzed cluster distribution across each decade (1950s–2020s)
- Identified top 5 clusters per decade

### Key Outcomes

```
Top 3 fear categories per cluster:

Shark & Aquatic Monster Horror (Cluster 1)
  Ecological / Natural Menace: 54 (74.0%)
  Invasion & Paranoia: 9 (12.3%)
  Body Horror: 5 (6.8%)

Modern Creature Feature Horror (Cluster 3)
  Invasion & Paranoia: 46 (40.4%)
  Grief & Familial Trauma: 13 (11.4%)
  Body Horror: 12 (10.5%)

Vintage Camp & Creature Schlock (1960s-80s) (Cluster 8)
  Invasion & Paranoia: 53 (34.4%)
  Captivity & Voyeuristic Sadism: 32 (20.8%)
  Possession & Loss of Agency: 27 (17.5%)

Modern Isolation & Survival Horror (Cluster 9)
  Invasion & Paranoia: 36 (35.3%)
  Grief & Familial Trauma: 14 (13.7%)
  Captivity & Voyeuristic Sadism: 12 (11.8%)

Classic Monster Movies (1950s-1960s) (Cluster 14)
  Invasion & Paranoia: 48 (46.6%)
  Body Horror: 28 (27.2%)
  Ecological / Natural Menace: 13 (12.6%)

Japanese Vintage Horror (Cluster 16)
  Grief & Familial Trauma: 36 (32.1%)
  Invasion & Paranoia: 19 (17.0%)
  Possession & Loss of Agency: 18 (16.1%)

Modern Female-Led Psychological Horror (Cluster 22)
  Invasion & Paranoia: 15 (21.7%)
  Captivity & Voyeuristic Sadism: 14 (20.3%)
  Possession & Loss of Agency: 10 (14.5%)

Rural / Backwoods Survival Horror (Cluster 25)
  Invasion & Paranoia: 27 (30.3%)
  Captivity & Voyeuristic Sadism: 27 (30.3%)
  Body Horror: 8 (9.0%)

Vintage Illustrated Monster Movies (Pre-1980) (Cluster 28)
  Invasion & Paranoia: 42 (37.2%)
  Body Horror: 19 (16.8%)
  Possession & Loss of Agency: 14 (12.4%)

Supernatural Gothic Horror (1990s-2000s) (Cluster 30)
  Invasion & Paranoia: 20 (40.8%)
  Captivity & Voyeuristic Sadism: 10 (20.4%)
  Possession & Loss of Agency: 6 (12.2%)

Latin American Horror (Vintage) (Cluster 31)
  Possession & Loss of Agency: 12 (26.7%)
  Invasion & Paranoia: 10 (22.2%)
  Body Horror: 7 (15.6%)

Modern Monster & Creature Horror (Cluster 39)
  Invasion & Paranoia: 32 (35.6%)
  Possession & Loss of Agency: 19 (21.1%)
  Captivity & Voyeuristic Sadism: 11 (12.2%)

Vintage Gothic (mix) (Cluster 41)
  Possession & Loss of Agency: 32 (21.3%)
  Invasion & Paranoia: 30 (20.0%)
  Captivity & Voyeuristic Sadism: 24 (16.0%)

Religious Horror (Cluster 42)
  Possession & Loss of Agency: 25 (49.0%)
  Institutional & Structural Control: 6 (11.8%)
  Invasion & Paranoia: 5 (9.8%)

Modern Neon / Synthwave (Cluster 43)
  Invasion & Paranoia: 17 (20.2%)
  Captivity & Voyeuristic Sadism: 17 (20.2%)
  Possession & Loss of Agency: 13 (15.5%)

B-Movie Creature Features & Kaiju (Cluster 46)
  Invasion & Paranoia: 40 (50.6%)
  Ecological / Natural Menace: 17 (21.5%)
  Body Horror: 12 (15.2%)

Modern Teen / Ensemble Horror-Comedy (Cluster 47)
  Invasion & Paranoia: 40 (38.8%)
  Captivity & Voyeuristic Sadism: 23 (22.3%)
  Possession & Loss of Agency: 14 (13.6%)

South Asian Horror (Cluster 48)
  Invasion & Paranoia: 17 (35.4%)
  Possession & Loss of Agency: 15 (31.2%)
  Grief & Familial Trauma: 8 (16.7%)


Top clusters per decade:

1950s (Total movies: 162)
  Cluster 14.0 (Classic Monster Movies (1950s-1960s)): 61 (37.7%)
  Cluster 16.0 (Japanese Vintage Horror): 43 (26.5%)
  Cluster 41.0 (Vintage Gothic (mix)): 22 (13.6%)
  Cluster 31.0 (Latin American Horror (Vintage)): 17 (10.5%)
  Cluster 46.0 (B-Movie Creature Features & Kaiju): 6 (3.7%)

1960s (Total movies: 140)
  Cluster 41.0 (Vintage Gothic (mix)): 43 (30.7%)
  Cluster 16.0 (Japanese Vintage Horror): 28 (20.0%)
  Cluster 14.0 (Classic Monster Movies (1950s-1960s)): 24 (17.1%)
  Cluster 31.0 (Latin American Horror (Vintage)): 16 (11.4%)
  Cluster 28.0 (Vintage Illustrated Monster Movies (Pre-1980)): 11 (7.9%)

1970s (Total movies: 156)
  Cluster 41.0 (Vintage Gothic (mix)): 50 (32.1%)
  Cluster 8.0 (Vintage Camp & Creature Schlock (1960s-80s)): 25 (16.0%)
  Cluster 28.0 (Vintage Illustrated Monster Movies (Pre-1980)): 20 (12.8%)
  Cluster 16.0 (Japanese Vintage Horror): 12 (7.7%)
  Cluster 14.0 (Classic Monster Movies (1950s-1960s)): 11 (7.1%)

1980s (Total movies: 168)
  Cluster 8.0 (Vintage Camp & Creature Schlock (1960s-80s)): 72 (42.9%)
  Cluster 28.0 (Vintage Illustrated Monster Movies (Pre-1980)): 25 (14.9%)
  Cluster 41.0 (Vintage Gothic (mix)): 14 (8.3%)
  Cluster 16.0 (Japanese Vintage Horror): 12 (7.1%)
  Cluster 39.0 (Modern Monster & Creature Horror): 9 (5.4%)

1990s (Total movies: 164)
  Cluster 8.0 (Vintage Camp & Creature Schlock (1960s-80s)): 39 (23.8%)
  Cluster 28.0 (Vintage Illustrated Monster Movies (Pre-1980)): 34 (20.7%)
  Cluster 30.0 (Supernatural Gothic Horror (1990s-2000s)): 18 (11.0%)
  Cluster 16.0 (Japanese Vintage Horror): 16 (9.8%)
  Cluster 39.0 (Modern Monster & Creature Horror): 14 (8.5%)

2000s (Total movies: 176)
  Cluster 3.0 (Modern Creature Feature Horror): 39 (22.2%)
  Cluster 39.0 (Modern Monster & Creature Horror): 20 (11.4%)
  Cluster 25.0 (Rural / Backwoods Survival Horror): 16 (9.1%)
  Cluster 30.0 (Supernatural Gothic Horror (1990s-2000s)): 14 (8.0%)
  Cluster 46.0 (B-Movie Creature Features & Kaiju): 13 (7.4%)

2010s (Total movies: 320)
  Cluster 25.0 (Rural / Backwoods Survival Horror): 45 (14.1%)
  Cluster 3.0 (Modern Creature Feature Horror): 42 (13.1%)
  Cluster 9.0 (Modern Isolation & Survival Horror): 38 (11.9%)
  Cluster 47.0 (Modern Teen / Ensemble Horror-Comedy): 30 (9.4%)
  Cluster 22.0 (Modern Female-Led Psychological Horror): 28 (8.8%)

2020s (Total movies: 342)
  Cluster 47.0 (Modern Teen / Ensemble Horror-Comedy): 66 (19.3%)
  Cluster 43.0 (Modern Neon / Synthwave): 46 (13.5%)
  Cluster 9.0 (Modern Isolation & Survival Horror): 46 (13.5%)
  Cluster 22.0 (Modern Female-Led Psychological Horror): 31 (9.1%)
  Cluster 1.0 (Shark & Aquatic Monster Horror): 27 (7.9%)

```


**Clusters × Fear Categories:**
Some clusters show strong alignment with specific fears, such as Shark & Aquatic Monster Horror (74% Ecological / Natural Menace), Religious Horror (49% Possession & Loss of Agency), and Classic Monster Movies (47% Invasion & Paranoia.)

Other clusters are more mixed, suggesting poster aesthetics respond to multiple types of fears.

**Clusters × Decades:**
Clear temporal patterns emerge, such as:
- **1950s-1960s** dominated by Classic Monster Movies, Japanese Vintage Horror, and Vintage Gothic
- **1980s** peak of Vintage Camp & Creature Schlock and Vintage Illustrated Monster Movies, for which the LLM-assisted labeling had suggested the time frames of "1960s-80s" and "Pre-1980", respectively.
- **2000s-2010s** shift to Modern Creature Feature Horror, Modern Monster & Creature Horror, and Rural Survival Horror
- **2020s** rise of Modern Teen Horror-Comedy (19%) and Modern Neon/Synthwave (14%)


## Phase 8 — Textual Interpretation of Visual Clusters (TF-IDF)

### Goal
To test whether visually coherent clusters correlate with a narrative/semantic theme.

### Process
- Grouped plot overviews by poster cluster
- Ran TF-IDF on cleaned text
- Re-ran with expanded stopwords to remove generic horror vocabulary

### Technical Decisions
- `doc_count ≥ 8` threshold in TF-IDF configuration to prioritize terms that recur across multiple films in a cluster—at the cost of sometimes returning no terms for clusters whose narratives are thematically diverse or primarily aesthetic.
- Dataset-specific stopword expansion (including words like "family, death, film, man, woman...")

Note: Expanding the stopword list revealed more distinctive keywords in some clusters, but resulted in no significant terms (at doc_count ≥ 8) for others (e.g., Modern Female-Led Psychological Horror).

### Key Outcomes


```
Cluster 0: (unlabeled cluster) (n=99)
    school                         TF-IDF sum: 1.7121  Doc count: 8
    revenge                        TF-IDF sum: 1.6618  Doc count: 10
    come                           TF-IDF sum: 1.3620  Doc count: 8
    sexual                         TF-IDF sum: 1.2131  Doc count: 8
    make                           TF-IDF sum: 1.0503  Doc count: 8
    beautiful                      TF-IDF sum: 0.9876  Doc count: 8

Cluster 1: Shark & Aquatic Monster Horror (n=73)
    shark                          TF-IDF sum: 3.5409  Doc count: 24
    sharks                         TF-IDF sum: 2.7317  Doc count: 18
    white                          TF-IDF sum: 1.8287  Doc count: 13
    great                          TF-IDF sum: 1.8287  Doc count: 13
    fight                          TF-IDF sum: 1.7860  Doc count: 12
    attack                         TF-IDF sum: 1.5094  Doc count: 8
    deadly                         TF-IDF sum: 1.5069  Doc count: 10
    water                          TF-IDF sum: 1.3856  Doc count: 9
    island                         TF-IDF sum: 1.3713  Doc count: 8
    sea                            TF-IDF sum: 1.3563  Doc count: 9

Cluster 2: (unlabeled cluster) (n=117)
    beautiful                      TF-IDF sum: 1.8211  Doc count: 12
    make                           TF-IDF sum: 1.5017  Doc count: 10
    body                           TF-IDF sum: 1.4000  Doc count: 8
    takes                          TF-IDF sum: 1.1200  Doc count: 9

Cluster 3: Modern Creature Feature Horror (n=114)
    monster                        TF-IDF sum: 1.5317  Doc count: 8
    supernatural                   TF-IDF sum: 1.4749  Doc count: 9
    creature                       TF-IDF sum: 1.4571  Doc count: 10
    ancient                        TF-IDF sum: 1.3616  Doc count: 10
    small                          TF-IDF sum: 1.3033  Doc count: 8

Cluster 4: (unlabeled cluster) (n=129)
    couple                         TF-IDF sum: 1.9889  Doc count: 11
    terrifying                     TF-IDF sum: 1.9469  Doc count: 12
    trapped                        TF-IDF sum: 1.4305  Doc count: 8
    escape                         TF-IDF sum: 1.3361  Doc count: 8

Cluster 5: (unlabeled cluster) (n=102)
    takes                          TF-IDF sum: 1.1644  Doc count: 8

Cluster 6: (unlabeled cluster) (n=41)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 7: (unlabeled cluster) (n=160)
    haunted                        TF-IDF sum: 2.0165  Doc count: 12
    killing                        TF-IDF sum: 2.0110  Doc count: 12
    horror                         TF-IDF sum: 1.8807  Doc count: 13
    creatures                      TF-IDF sum: 1.7373  Doc count: 9
    mansion                        TF-IDF sum: 1.6393  Doc count: 9
    dr                             TF-IDF sum: 1.5756  Doc count: 10
    series                         TF-IDF sum: 1.5037  Doc count: 10
    small                          TF-IDF sum: 1.4799  Doc count: 10
    sister                         TF-IDF sum: 1.4490  Doc count: 8
    brother                        TF-IDF sum: 1.4061  Doc count: 8
    just                           TF-IDF sum: 1.3962  Doc count: 9
    school                         TF-IDF sum: 1.3887  Doc count: 9
    dark                           TF-IDF sum: 1.3647  Doc count: 9
    monster                        TF-IDF sum: 1.3344  Doc count: 8
    things                         TF-IDF sum: 1.3214  Doc count: 9

Cluster 8: Vintage Camp & Creature Schlock (1960s-80s) (n=154)
    school                         TF-IDF sum: 2.2416  Doc count: 9
    high                           TF-IDF sum: 1.8294  Doc count: 11
    island                         TF-IDF sum: 1.6433  Doc count: 8
    monster                        TF-IDF sum: 1.6160  Doc count: 8
    students                       TF-IDF sum: 1.5563  Doc count: 8
    police                         TF-IDF sum: 1.5053  Doc count: 9
    spirit                         TF-IDF sum: 1.4123  Doc count: 8
    team                           TF-IDF sum: 1.4094  Doc count: 10
    earth                          TF-IDF sum: 1.3434  Doc count: 8
    city                           TF-IDF sum: 1.2835  Doc count: 9
    series                         TF-IDF sum: 1.2430  Doc count: 9
    forces                         TF-IDF sum: 1.2218  Doc count: 8
    kill                           TF-IDF sum: 1.2016  Doc count: 8
    body                           TF-IDF sum: 1.1852  Doc count: 8
    stop                           TF-IDF sum: 1.1732  Doc count: 9

Cluster 9: Modern Isolation & Survival Horror (n=102)
    woods                          TF-IDF sum: 2.2558  Doc count: 14
    forest                         TF-IDF sum: 1.6562  Doc count: 8
    story                          TF-IDF sum: 1.3779  Doc count: 9
    events                         TF-IDF sum: 1.3314  Doc count: 9
    deep                           TF-IDF sum: 1.3214  Doc count: 8
    trip                           TF-IDF sum: 1.2572  Doc count: 8
    year                           TF-IDF sum: 1.2401  Doc count: 8
    small                          TF-IDF sum: 1.2295  Doc count: 8
    lives                          TF-IDF sum: 1.2148  Doc count: 8
    long                           TF-IDF sum: 1.1898  Doc count: 8

Cluster 10: (unlabeled cluster) (n=91)
    school                         TF-IDF sum: 1.9538  Doc count: 15
    student                        TF-IDF sum: 1.5948  Doc count: 11
    human                          TF-IDF sum: 1.5605  Doc count: 12
    stop                           TF-IDF sum: 1.2761  Doc count: 9
    high                           TF-IDF sum: 1.2121  Doc count: 11
    day                            TF-IDF sum: 1.0723  Doc count: 11
    humans                         TF-IDF sum: 1.0259  Doc count: 8

Cluster 11: (unlabeled cluster) (n=104)
    dark                           TF-IDF sum: 1.7485  Doc count: 13
    blood                          TF-IDF sum: 1.5673  Doc count: 10
    lives                          TF-IDF sum: 1.5025  Doc count: 8
    long                           TF-IDF sum: 1.3727  Doc count: 9

Cluster 12: (unlabeled cluster) (n=144)
    haunted                        TF-IDF sum: 2.0801  Doc count: 13
    horror                         TF-IDF sum: 2.0479  Doc count: 15
    lives                          TF-IDF sum: 1.7978  Doc count: 12
    small                          TF-IDF sum: 1.6801  Doc count: 11
    couple                         TF-IDF sum: 1.5686  Doc count: 8
    remote                         TF-IDF sum: 1.4808  Doc count: 9
    dark                           TF-IDF sum: 1.4325  Doc count: 10
    isolated                       TF-IDF sum: 1.4042  Doc count: 8
    school                         TF-IDF sum: 1.3456  Doc count: 9
    students                       TF-IDF sum: 1.3060  Doc count: 8
    inside                         TF-IDF sum: 1.2742  Doc count: 8
    moves                          TF-IDF sum: 1.2455  Doc count: 8
    year                           TF-IDF sum: 1.2247  Doc count: 9
    starts                         TF-IDF sum: 1.1833  Doc count: 9
    set                            TF-IDF sum: 1.1589  Doc count: 8

Cluster 13: (unlabeled cluster) (n=112)
    small                          TF-IDF sum: 1.5907  Doc count: 12
    story                          TF-IDF sum: 1.4772  Doc count: 9
    lives                          TF-IDF sum: 1.2411  Doc count: 8
    goes                           TF-IDF sum: 1.2404  Doc count: 8
    war                            TF-IDF sum: 1.1953  Doc count: 8
    police                         TF-IDF sum: 1.1054  Doc count: 8
    dark                           TF-IDF sum: 1.0914  Doc count: 8

Cluster 14: Classic Monster Movies (1950s-1960s) (n=103)
    local                          TF-IDF sum: 2.2204  Doc count: 11
    earth                          TF-IDF sum: 2.1892  Doc count: 15
    scientist                      TF-IDF sum: 2.1233  Doc count: 14
    giant                          TF-IDF sum: 2.1019  Doc count: 15
    monster                        TF-IDF sum: 1.9524  Doc count: 12
    creature                       TF-IDF sum: 1.7937  Doc count: 11
    doctor                         TF-IDF sum: 1.6462  Doc count: 10
    remote                         TF-IDF sum: 1.3987  Doc count: 9
    discovered                     TF-IDF sum: 1.3718  Doc count: 9
    alien                          TF-IDF sum: 1.3691  Doc count: 8
    dr                             TF-IDF sum: 1.3605  Doc count: 8
    small                          TF-IDF sum: 1.3341  Doc count: 10
    deaths                         TF-IDF sum: 1.3077  Doc count: 8
    scientists                     TF-IDF sum: 1.2968  Doc count: 10
    destroy                        TF-IDF sum: 1.2253  Doc count: 9

Cluster 15: (unlabeled cluster) (n=80)
    spirit                         TF-IDF sum: 1.4515  Doc count: 8
    curse                          TF-IDF sum: 1.1416  Doc count: 8

Cluster 16: Japanese Vintage Horror (n=112)
    ghost                          TF-IDF sum: 3.4552  Doc count: 29
    killed                         TF-IDF sum: 1.7617  Doc count: 12
    cat                            TF-IDF sum: 1.7559  Doc count: 8
    love                           TF-IDF sum: 1.6614  Doc count: 14
    beautiful                      TF-IDF sum: 1.4820  Doc count: 9
    story                          TF-IDF sum: 1.4711  Doc count: 8
    takes                          TF-IDF sum: 1.4188  Doc count: 11
    brother                        TF-IDF sum: 1.4089  Doc count: 8
    samurai                        TF-IDF sum: 1.3771  Doc count: 8
    series                         TF-IDF sum: 1.1965  Doc count: 8
    revenge                        TF-IDF sum: 1.1827  Doc count: 10
    falls                          TF-IDF sum: 1.1699  Doc count: 10
    goes                           TF-IDF sum: 1.1696  Doc count: 10
    suicide                        TF-IDF sum: 1.1001  Doc count: 8
    spirit                         TF-IDF sum: 1.0990  Doc count: 8

Cluster 17: (unlabeled cluster) (n=62)
    school                         TF-IDF sum: 1.5415  Doc count: 11
    year                           TF-IDF sum: 1.3031  Doc count: 8
    love                           TF-IDF sum: 1.1129  Doc count: 9
    blood                          TF-IDF sum: 1.0232  Doc count: 8

Cluster 18: (unlabeled cluster) (n=142)
    scientist                      TF-IDF sum: 1.8361  Doc count: 12
    dr                             TF-IDF sum: 1.8110  Doc count: 13
    victims                        TF-IDF sum: 1.5285  Doc count: 8
    murders                        TF-IDF sum: 1.4736  Doc count: 9
    small                          TF-IDF sum: 1.4108  Doc count: 9
    story                          TF-IDF sum: 1.4068  Doc count: 8
    horror                         TF-IDF sum: 1.3692  Doc count: 10
    body                           TF-IDF sum: 1.3489  Doc count: 8
    living                         TF-IDF sum: 1.3219  Doc count: 9
    comes                          TF-IDF sum: 1.2640  Doc count: 8
    remote                         TF-IDF sum: 1.2606  Doc count: 8

Cluster 19: (unlabeled cluster) (n=82)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 20: (unlabeled cluster) (n=88)
    horror                         TF-IDF sum: 1.3289  Doc count: 9
    trapped                        TF-IDF sum: 1.2264  Doc count: 8
    crew                           TF-IDF sum: 1.1785  Doc count: 8

Cluster 21: (unlabeled cluster) (n=138)
    island                         TF-IDF sum: 2.3941  Doc count: 12
    war                            TF-IDF sum: 2.2143  Doc count: 15
    fight                          TF-IDF sum: 2.1477  Doc count: 12
    survivors                      TF-IDF sum: 1.9648  Doc count: 13
    zombies                        TF-IDF sum: 1.9468  Doc count: 12
    city                           TF-IDF sum: 1.8643  Doc count: 10
    army                           TF-IDF sum: 1.8069  Doc count: 13
    soldiers                       TF-IDF sum: 1.7867  Doc count: 12
    save                           TF-IDF sum: 1.7846  Doc count: 13
    small                          TF-IDF sum: 1.7517  Doc count: 12
    virus                          TF-IDF sum: 1.7452  Doc count: 9
    zombie                         TF-IDF sum: 1.7382  Doc count: 11
    human                          TF-IDF sum: 1.6137  Doc count: 11
    american                       TF-IDF sum: 1.6113  Doc count: 10
    mission                        TF-IDF sum: 1.5628  Doc count: 11

Cluster 22: Modern Female-Led Psychological Horror (n=69)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 23: (unlabeled cluster) (n=110)
    serial                         TF-IDF sum: 1.4871  Doc count: 8
    lives                          TF-IDF sum: 1.4173  Doc count: 8
    blood                          TF-IDF sum: 1.3485  Doc count: 8
    tries                          TF-IDF sum: 1.3012  Doc count: 8
    day                            TF-IDF sum: 1.2120  Doc count: 8

Cluster 24: (unlabeled cluster) (n=113)
    dark                           TF-IDF sum: 1.9674  Doc count: 16
    friend                         TF-IDF sum: 1.4541  Doc count: 8
    face                           TF-IDF sum: 1.2670  Doc count: 8
    haunted                        TF-IDF sum: 1.2270  Doc count: 8
    year                           TF-IDF sum: 1.0824  Doc count: 9

Cluster 25: Rural / Backwoods Survival Horror (n=89)
    remote                         TF-IDF sum: 1.2850  Doc count: 9

Cluster 26: (unlabeled cluster) (n=118)
    tries                          TF-IDF sum: 1.5951  Doc count: 8
    student                        TF-IDF sum: 1.4650  Doc count: 8
    story                          TF-IDF sum: 1.3578  Doc count: 9
    love                           TF-IDF sum: 1.3460  Doc count: 9
    lives                          TF-IDF sum: 1.2998  Doc count: 8
    lover                          TF-IDF sum: 1.1398  Doc count: 8

Cluster 27: (unlabeled cluster) (n=104)
    apartment                      TF-IDF sum: 1.4168  Doc count: 8
    set                            TF-IDF sum: 1.2586  Doc count: 8
    gets                           TF-IDF sum: 1.2476  Doc count: 8

Cluster 28: Vintage Illustrated Monster Movies (Pre-1980) (n=113)
    scientist                      TF-IDF sum: 1.5197  Doc count: 9
    alien                          TF-IDF sum: 1.4816  Doc count: 9
    space                          TF-IDF sum: 1.3939  Doc count: 10
    named                          TF-IDF sum: 1.1430  Doc count: 8
    just                           TF-IDF sum: 1.0648  Doc count: 9

Cluster 29: (unlabeled cluster) (n=112)
    child                          TF-IDF sum: 1.9140  Doc count: 10
    couple                         TF-IDF sum: 1.6465  Doc count: 10
    takes                          TF-IDF sum: 1.5183  Doc count: 8
    small                          TF-IDF sum: 1.2464  Doc count: 8
    escape                         TF-IDF sum: 1.2151  Doc count: 8

Cluster 30: Supernatural Gothic Horror (1990s-2000s) (n=49)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 31: Latin American Horror (Vintage) (n=45)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 32: (unlabeled cluster) (n=74)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 33: (unlabeled cluster) (n=109)
    escape                         TF-IDF sum: 1.5259  Doc count: 10
    team                           TF-IDF sum: 1.3765  Doc count: 9
    deadly                         TF-IDF sum: 1.3479  Doc count: 9

Cluster 34: (unlabeled cluster) (n=169)
    zombies                        TF-IDF sum: 2.8433  Doc count: 15
    horror                         TF-IDF sum: 2.0195  Doc count: 11
    survive                        TF-IDF sum: 1.9381  Doc count: 13
    fight                          TF-IDF sum: 1.9327  Doc count: 14
    survival                       TF-IDF sum: 1.7350  Doc count: 10
    turns                          TF-IDF sum: 1.7287  Doc count: 12
    virus                          TF-IDF sum: 1.6799  Doc count: 10
    couple                         TF-IDF sum: 1.6724  Doc count: 9
    survivors                      TF-IDF sum: 1.6349  Doc count: 8
    city                           TF-IDF sum: 1.6050  Doc count: 10
    weekend                        TF-IDF sum: 1.5872  Doc count: 8
    remote                         TF-IDF sum: 1.5841  Doc count: 9
    lives                          TF-IDF sum: 1.5715  Doc count: 9
    team                           TF-IDF sum: 1.5653  Doc count: 9
    small                          TF-IDF sum: 1.5456  Doc count: 9

Cluster 35: (unlabeled cluster) (n=80)
    stop                           TF-IDF sum: 1.2919  Doc count: 9
    school                         TF-IDF sum: 1.1997  Doc count: 8

Cluster 36: (unlabeled cluster) (n=103)
    love                           TF-IDF sum: 1.4370  Doc count: 9
    scientist                      TF-IDF sum: 1.3894  Doc count: 9
    dr                             TF-IDF sum: 1.3433  Doc count: 8
    american                       TF-IDF sum: 1.2269  Doc count: 8

Cluster 37: (unlabeled cluster) (n=68)
    story                          TF-IDF sum: 1.2104  Doc count: 9
    love                           TF-IDF sum: 1.1627  Doc count: 9

Cluster 38: (unlabeled cluster) (n=118)
    remote                         TF-IDF sum: 1.9965  Doc count: 14
    couple                         TF-IDF sum: 1.4587  Doc count: 8
    turns                          TF-IDF sum: 1.4009  Doc count: 8
    sinister                       TF-IDF sum: 1.3983  Doc count: 8
    fight                          TF-IDF sum: 1.3501  Doc count: 8
    events                         TF-IDF sum: 1.3051  Doc count: 8
    terrifying                     TF-IDF sum: 1.2345  Doc count: 9
    just                           TF-IDF sum: 1.2194  Doc count: 10

Cluster 39: Modern Monster & Creature Horror (n=90)
    demon                          TF-IDF sum: 1.6744  Doc count: 9
    murders                        TF-IDF sum: 1.1568  Doc count: 8
    haunted                        TF-IDF sum: 1.0890  Doc count: 8

Cluster 40: (unlabeled cluster) (n=68)
    day                            TF-IDF sum: 0.9874  Doc count: 8

Cluster 41: Vintage Gothic (mix) (n=150)
    vampire                        TF-IDF sum: 2.8603  Doc count: 19
    blood                          TF-IDF sum: 2.6080  Doc count: 19
    dr                             TF-IDF sum: 2.4047  Doc count: 17
    castle                         TF-IDF sum: 2.3474  Doc count: 11
    count                          TF-IDF sum: 2.2115  Doc count: 12
    doctor                         TF-IDF sum: 1.9209  Doc count: 13
    dracula                        TF-IDF sum: 1.8532  Doc count: 10
    scientist                      TF-IDF sum: 1.6806  Doc count: 8
    body                           TF-IDF sum: 1.6558  Doc count: 11
    murders                        TF-IDF sum: 1.5557  Doc count: 10
    human                          TF-IDF sum: 1.5483  Doc count: 9
    horror                         TF-IDF sum: 1.5076  Doc count: 8
    small                          TF-IDF sum: 1.4282  Doc count: 9
    th                             TF-IDF sum: 1.4050  Doc count: 8
    village                        TF-IDF sum: 1.4031  Doc count: 8

Cluster 42: Religious Horror (n=51)
    priest                         TF-IDF sum: 1.7524  Doc count: 12
    dark                           TF-IDF sum: 1.2309  Doc count: 9

Cluster 43: Modern Neon / Synthwave (n=84)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 44: (unlabeled cluster) (n=80)
    haunted                        TF-IDF sum: 1.3960  Doc count: 8

Cluster 45: (unlabeled cluster) (n=119)
    lives                          TF-IDF sum: 1.4389  Doc count: 10
    series                         TF-IDF sum: 1.2444  Doc count: 8
    like                           TF-IDF sum: 1.2287  Doc count: 10

Cluster 46: B-Movie Creature Features & Kaiju (n=79)
    alien                          TF-IDF sum: 2.0616  Doc count: 15
    planet                         TF-IDF sum: 1.9478  Doc count: 9
    island                         TF-IDF sum: 1.8790  Doc count: 8
    monster                        TF-IDF sum: 1.5989  Doc count: 11
    small                          TF-IDF sum: 1.5504  Doc count: 10
    team                           TF-IDF sum: 1.3987  Doc count: 8
    crash                          TF-IDF sum: 1.3779  Doc count: 9
    stop                           TF-IDF sum: 1.3033  Doc count: 9

Cluster 47: Modern Teen / Ensemble Horror-Comedy (n=103)
    school                         TF-IDF sum: 2.0246  Doc count: 12
    fight                          TF-IDF sum: 1.5406  Doc count: 8
    small                          TF-IDF sum: 1.3983  Doc count: 9

Cluster 48: South Asian Horror (n=48)
  (no terms with doc_count ≥ 8 after expanded stopwords)

Cluster 49: (unlabeled cluster) (n=64)
  (no terms with doc_count ≥ 8 after expanded stopwords)

```

Clusters with distinctive visual motifs produce clear, cohesive narrative signals. Cases like Shark & Aquatic Monster Horror (Cluster 1), Classic Monster Movies (Cluster 14), Vintage Gothic (Cluster 41), or Creature Features & Kaiju (Cluster 46) show TF-IDF validation. In these cases, visual iconography directly corresponds to the narrative content.

TF-IDF also proves valuable for discovering themes in unlabeled clusters. Cluster 34, for instance, was not manually labeled in the previous inspection, but TF-IDF suggests it centers on zombie and survival narratives—with "zombies," "virus," "survive," and "survivors" as the most representative terms.

Several other clusters fail this narrative validation. In cases where clusters respond more to aesthetics than thematic tropes, such as Modern Neon/Synthwave (Cluster 43), TF-IDF does not return relevant keywords when using the extended stopwords list. Other clusters seem to blend multiple tropes without thematic unity.



## Conclusion and Future Directions

This project explored whether horror films reveal patterns in collective fear by combining metadata analysis, LLM-assisted classification, and unsupervised visual clustering. The hybrid approach proved promising in uncovering narrative and visual patterns across seven decades of horror cinema. If fear categories rise and fall in ways that correlate with historical shocks within regions, that would support the cultural mirror hypothesis.

To truly test whether horror mirrors cultural contexts, this pipeline should be expanded to conduct region-specific and/or country-specific analysis. By isolating cases geographically and temporally, we could examine whether fear patterns align with distinct historical moments (Cold War anxieties, authoritarian regimes, economic collapse, pandemic fears, etc). Combining both textual (plot, keywords, fear categories) and visual (poster iconography, color palettes, composition) information at a granular level would allow us to pinpoint how local contexts shape horror's visual and narrative language, transforming this exploratory study into a tool for cultural and social studies.
